# Fine-tuning de Qwen2.5-3B-Instruct en el formato de la app (tercera versión)

Notebook de Colab (GPU T4 gratuita). Es la segunda versión del fine-tuning QLoRA:
la primera (`finetuning_qwen_colab.ipynb`) entrenó al modelo a devolver un único
JSON con un prompt propio y de un solo turno, pero la aplicación lo usa de otra
forma (dos llamadas por mensaje, con los prompts reales, herramientas nativas de
Qwen e historial). Servido con Ollama rendía 91% de tool calling con el prompt de
entrenamiento y entre 61% y 76% por el camino de la app. Ver
`finetuning/DEVLOG.md`, "Integración con Ollama" y "Segunda versión del dataset".

Esta versión entrena con conversaciones idénticas a las que arma la app:
`finetuning/armar_dataset_app.py` importa los prompts y las herramientas de
`app/` y los renderiza con la misma función que usa la inferencia
(`app.agents.llm.a_formato_qwen`).

**Antes de correr esto**: `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución`
→ GPU T4.

Qué hace, en orden:
1. Clona el repo, regenera el dataset base (MASSIVE + MULTI3NLU++ + manual) y lo
   convierte al formato de la app.
2. Instala Unsloth y carga Qwen2.5-3B-Instruct en 4 bits.
3. Agrega adaptadores LoRA con los mismos hiperparámetros de la tesis.
4. Entrena 2 épocas (ver la celda de entrenamiento por qué no 3).
5. Exporta a GGUF para servir con Ollama.
6. Mide tool calling sobre el conjunto de prueba, con el prompt real del ejecutor.

La medición que vale para la tesis se hace después, en local, con el GGUF servido
por Ollama: `python -m scripts.evaluar_modelo_local`.

**Tercera versión del dataset.** Suma las pautas generales del organismo (`conocimiento/pautas_generales/`) en el 40 % de los prompts del ejecutor, conversaciones donde aplicarlas cambia la respuesta (emergencias: primero el número de emergencias; pedidos de opinión política o de claves) y la grilla de horarios de cada agenda en las reservas. Las conversaciones más largas rondan los 2450 tokens, dentro de `MAX_SEQ_LENGTH`. Ver `finetuning/DEVLOG.md`, "Tercera versión del dataset".

## 1. Clonar el repo y regenerar el dataset

**El repo es privado** — un `git clone` anónimo devuelve 404. Dos opciones:

- **Dejarlo privado (recomendado mientras es tesis en progreso)**: crear un GitHub Personal Access Token de solo lectura (GitHub → Settings → Developer settings → Personal access tokens → Fine-grained, con acceso de lectura a `Schonp/OmniCivitas-PFI`) y guardarlo en Colab como secreto: ícono de llave 🔑 en la barra izquierda → "Agregar secreto nuevo" → nombre `GITHUB_TOKEN`, activar "Acceso al notebook". La celda de abajo lo usa automáticamente si existe.
- **Hacer el repo público**: si se decide eso más adelante, la misma celda funciona igual sin configurar nada — el token queda sin uso.

Esto es independiente de cómo abriste este notebook (subido a mano o desde github.com/.../blob/...): ese paso solo trae el archivo .ipynb, no le da a la máquina virtual de Colab ningún permiso sobre el repo.

In [ ]:
import os

try:
    from google.colab import userdata
    _token = userdata.get("GITHUB_TOKEN")
except Exception:
    _token = None

_repo = (
    f"https://{_token}@github.com/Schonp/OmniCivitas-PFI.git"
    if _token
    else "https://github.com/Schonp/OmniCivitas-PFI.git"
)

if not os.path.isdir("/content/OmniCivitas"):
    !git clone {_repo} /content/OmniCivitas

%cd /content/OmniCivitas/finetuning
!pip install -q -r requirements.txt
# armar_dataset_app.py importa los prompts y las herramientas de app/: hacen
# falta las dependencias de la app (solo para armar el dataset, no para entrenar).
!pip install -q -r ../requirements.txt

`data/` está en `.gitignore` (son archivos generados, no fuente) — por eso se
regeneran acá en vez de subirse a mano. `ejemplos_tool_calling.jsonl` y
`ejemplos_generados.jsonl` sí viajan con el repo, son la parte escrita/generada
a mano que ningún script puede reconstruir.

In [ ]:
!python preparar_massive.py
!python preparar_multi3nlu.py
!python generar_ejemplos.py
!python armar_dataset.py
# Conversión al formato de la app: se corre como módulo desde la raíz del repo
# para que `import app...` resuelva.
!cd .. && python -m finetuning.armar_dataset_app

## 1.5. Montar Google Drive (para no perder el entrenamiento)

La VM de Colab es efímera: si la sesión se desconecta (por inactividad, por el límite de la cuenta gratis, o porque se corta la conexión mientras no estás mirando), todo lo que haya quedado solo en `/content` se pierde con ella — pasó una vez entrenando toda la noche sin esto montado. Google Drive sí persiste entre sesiones, así que los checkpoints y el resultado final se guardan ahí en vez de en el disco local de la VM.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/omnicivitas-finetune"
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Los checkpoints y el resultado final se van a guardar en: {DRIVE_DIR}")


## 2. Instalar Unsloth

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2


## 3. Cargar Qwen2.5-3B-Instruct en 4 bits

In [ ]:
from unsloth import FastLanguageModel
import torch

# Las conversaciones del ejecutor llevan el prompt real, los esquemas de las
# herramientas y hasta seis turnos: llegan a unos 1900 tokens. 2560 deja margen.
MAX_SEQ_LENGTH = 2560

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

## 4. Adaptadores LoRA

Hiperparámetros tal como quedaron comprometidos en `chapter04.tex` ("Entrenamiento del modelo de lenguaje"): cuantización NF4 4 bits (ya aplicada al cargar el modelo arriba), rank 16, alpha 32, dropout 0.05, sobre las proyecciones de atención (`q/k/v/o_proj`) y feed-forward (`gate/up/down_proj`).

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)


## 5. Dataset → conversaciones

A diferencia de la primera versión, acá no hay que convertir nada: cada fila de
`app_train.jsonl` ya es una conversación (`messages`) con los roles system / user /
assistant, tal como la arma la app. Las llamadas a herramientas van como bloques
`<tool_call>` en el turno del asistente y los resultados como `<tool_response>` en
un turno del usuario, que es como la plantilla de Qwen2.5 representa el rol `tool`.

Hay dos tipos de conversación (campo `tipo`): `intake` (clasificación de
intención, responde el JSON de `ClasificacionIntent`) y `ejecutor` (herramientas
y respuesta al ciudadano).

In [ ]:
import json


def cargar_jsonl(ruta):
    return [json.loads(linea) for linea in open(ruta, encoding="utf-8")]


def a_conversacion(fila):
    return {"conversations": fila["messages"]}

In [ ]:
from collections import Counter

from datasets import Dataset

train_crudo = cargar_jsonl("data/app_train.jsonl")
val_crudo = cargar_jsonl("data/app_validacion.jsonl")
print("train:", Counter(f["tipo"] for f in train_crudo))
print("validacion:", Counter(f["tipo"] for f in val_crudo))

dataset_train = Dataset.from_list([a_conversacion(f) for f in train_crudo])
dataset_val = Dataset.from_list([a_conversacion(f) for f in val_crudo])


def formatting_prompts_func(examples):
    convos = examples["conversations"]
    textos = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in convos
    ]
    return {"text": textos}


dataset_train = dataset_train.map(formatting_prompts_func, batched=True)
dataset_val = dataset_val.map(formatting_prompts_func, batched=True)

largos = sorted(len(tokenizer(t)["input_ids"]) for t in dataset_train["text"])
print(f"tokens por conversación: mediana {largos[len(largos) // 2]}, máximo {largos[-1]}")
assert largos[-1] <= MAX_SEQ_LENGTH, "hay conversaciones que se cortarían: subir MAX_SEQ_LENGTH"
print("--- ejemplo ---")
print(dataset_train[0]["text"])

## 6. Entrenamiento

Mismos hiperparámetros que la primera versión (LR 2e-4, coseno, batch efectivo 16),
salvo las épocas: **2 en vez de 3**. Cada conversación es ahora 3 a 5 veces más
larga (el prompt real del ejecutor con las herramientas), así que el mismo número
de épocas pasaría las 6 horas de la primera corrida y el límite de la GPU gratuita.
Con 2 épocas el modelo ve cada conversación dos veces, sobre un dataset que tiene
mucha menos redundancia que los 13 mil ejemplos de MASSIVE de la primera versión.

Como antes, la pérdida se calcula solo sobre los turnos del asistente: lo que dice
el ciudadano y los resultados de las herramientas no se aprenden, se condicionan.

Al terminar, la misma celda guarda los adaptadores y exporta el GGUF a Drive (el paso 7 del notebook anterior), para que una corrida de noche no dependa de ejecutar otra celda.

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq

BATCH_MICRO = 4
GRAD_ACUM = 4  # 4 x 4 = 16

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_train,
    eval_dataset=dataset_val,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=BATCH_MICRO,
        gradient_accumulation_steps=GRAD_ACUM,
        num_train_epochs=2,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        optim="adamw_8bit",
        weight_decay=0.001,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        seed=3407,
        output_dir=f"{DRIVE_DIR}/checkpoints_app",  # en Drive: sobrevive a un corte de sesión
        report_to="none",
    ),
)

# Enmascara la pérdida sobre el prompt (system+user): sólo se entrena a predecir
# el turno del asistente, no a memorizar la instrucción del ciudadano.
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

trainer_stats = trainer.train()

# --- 7. Guardar y exportar a GGUF (para Ollama) ---
# En la misma celda que el entrenamiento: si se deja corriendo de noche, el
# guardado se hace apenas termina, sin esperar a que alguien ejecute otra celda
# antes de que Colab corte la sesión por inactividad.
# Se guarda directo en Drive (DRIVE_DIR), no en /content: si la VM se
# desconecta después de esta celda, el resultado ya está a salvo. Carpetas
# distintas a las de la primera versión, para no pisar ese modelo.
model.save_pretrained(f"{DRIVE_DIR}/qwen_omnicivitas_app_lora")
tokenizer.save_pretrained(f"{DRIVE_DIR}/qwen_omnicivitas_app_lora")

model.save_pretrained_gguf(
    f"{DRIVE_DIR}/qwen_omnicivitas_app_gguf", tokenizer, quantization_method="q4_k_m"
)

print(f"Listo en Google Drive: {DRIVE_DIR}/qwen_omnicivitas_app_gguf/ -- ahí está el .gguf para Ollama.")

## 7. Evaluación rápida sobre el held-out de tool calling

Las 67 instrucciones de `prueba_tool_calling.jsonl` (nunca vistas en entrenamiento
ni validación), con el prompt real del ejecutor y las herramientas que la app
ofrece para esa intención. Se informan dos números:

- **Exacto**: herramienta y todos los parámetros iguales (la métrica del notebook
  anterior).
- **Sin texto libre**: herramienta y parámetros estructurados iguales, sin comparar
  `descripcion` ni `referencia`, donde una paráfrasis válida no es un error.

La exactitud de intención la mide el intake, que es otra llamada: se evalúa en
local con `scripts/evaluar_modelo_local.py`, junto con todo lo demás, sobre el
GGUF servido por Ollama.

In [ ]:
import re

FastLanguageModel.for_inference(model)
TOOL_CALL = re.compile(r"<tool_call>\s*(\{.*?\})\s*(?:</tool_call>|$)", re.DOTALL)
CAMPOS_LIBRES = {"descripcion", "referencia"}


def generar(mensajes):
    texto = tokenizer.apply_chat_template(mensajes, tokenize=False, add_generation_prompt=True)
    entradas = tokenizer(texto, return_tensors="pt").to("cuda")
    salida = model.generate(**entradas, max_new_tokens=300, do_sample=False)
    return tokenizer.decode(salida[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True)


def primera_llamada(texto):
    """La primera llamada del texto, como {"nombre", "parametros"}; None si no hay.
    Mismo criterio que app/agents/llm.py (leer_respuesta), sin los formatos de
    respaldo: acá interesa medir si el modelo aprendió el formato de Qwen."""
    for bloque in TOOL_CALL.findall(texto):
        try:
            datos = json.loads(bloque)
        except json.JSONDecodeError:
            continue
        nombre = datos.get("name")
        if nombre == "derivar_a_operador":
            return None
        return {"nombre": nombre, "parametros": datos.get("arguments", {})}
    return None


def estructurados(llamada):
    if not llamada:
        return None
    parametros = {
        k: str(v).strip().lower()
        for k, v in (llamada.get("parametros") or {}).items()
        if k not in CAMPOS_LIBRES
    }
    return llamada.get("nombre"), tuple(sorted(parametros.items()))

In [ ]:
prueba = cargar_jsonl("data/app_prueba.jsonl")

exactos = 0
sin_texto_libre = 0
detalle = []

for ejemplo in prueba:
    crudo = generar(ejemplo["messages"])
    pred = primera_llamada(crudo)
    esperado = ejemplo["esperado"]
    exacto = pred == esperado
    estructurado = estructurados(pred) == estructurados(esperado)
    exactos += int(exacto)
    sin_texto_libre += int(estructurado)
    detalle.append({
        "instruccion": ejemplo["instruccion"],
        "esperado": esperado,
        "pred": pred,
        "crudo": crudo,
        "exacto": exacto,
        "sin_texto_libre": estructurado,
    })

total = len(prueba)
print(f"Total evaluado: {total}")
print(f"Tool calling exacto: {exactos / total:.1%} ({exactos}/{total})")
print(f"Tool calling sin campos de texto libre: {sin_texto_libre / total:.1%} ({sin_texto_libre}/{total})")

# El detalle completo (con la salida cruda de cada ejemplo) queda en Drive, para
# diagnosticar cualquier fallo sin tener que regenerar nada.
with open(f"{DRIVE_DIR}/evaluacion_app.jsonl", "w", encoding="utf-8") as archivo:
    for fila in detalle:
        archivo.write(json.dumps(fila, ensure_ascii=False) + "\n")
print("Fallos:")
for fila in detalle:
    if not fila["sin_texto_libre"]:
        print("-", fila["instruccion"], "| esperado:", fila["esperado"], "| pred:", fila["pred"])

## Próximo paso (fuera de este notebook)

1. Bajar de Drive `omnicivitas-finetune/qwen_omnicivitas_app_gguf/*.gguf` y dejarlo
   en `ollama/` del repo con el nombre que espera `ollama/Modelfile`
   (`Qwen2.5-3B-Instruct.Q4_K_M.gguf`). Conviene renombrar antes el de la versión
   anterior (por ejemplo a `...v2.gguf`) para poder volver a él.
2. `ollama create omnicivitas-qwen -f ollama/Modelfile`.
3. Medir por el camino de la app: `python -m scripts.evaluar_modelo_local`.